# Phase 1: Environment Setup & Dataset Preparation
**Project:** Efficient Chain-of-Thought (CoT) Distillation in Small Language Models
**Execution Environment:** Kaggle GPU Tier (Dual T4 or Single P100)

In [ ]:
# Step 1: Install required dependencies on Kaggle GPU
!pip install -q unsloth trl transformers datasets peft bitsandbytes accelerate rouge_score evaluate pyyaml

In [ ]:
# Step 2: Configure System Path & Load Modules
import sys
import os
import yaml

# Ensure project root is in python path
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data_utils import (
    setup_environment,
    load_and_filter_dataset,
    prepare_splits,
    save_splits,
    compute_token_stats
)

print("Project Root:", PROJECT_ROOT)

In [ ]:
# Step 3: Load Configuration
config_path = os.path.join(PROJECT_ROOT, "configs", "qlora_config.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

print("Configuration loaded successfully.")

In [ ]:
# Step 4: Setup Environment and Seeds
env_info = setup_environment(seed=config['project']['seed'])

In [ ]:
# Step 5: Download & Filter Reasoning Dataset
raw_ds = load_and_filter_dataset(
    dataset_name=config['data']['dataset_name'],
    fallback_dataset_name=config['data']['fallback_dataset_name'],
    n_samples=config['data']['n_samples'],
    seed=config['project']['seed']
)

In [ ]:
# Step 6: Convert to Chat Template & Create Train/Test Splits
train_ds, test_ds = prepare_splits(
    dataset=raw_ds,
    train_ratio=config['data']['train_ratio'],
    seed=config['project']['seed'],
    model_type=config['model']['model_type']
)

In [ ]:
# Step 7: Inspect Sample Processed Record
print("=== Sample Formatted Entry ===")
print(train_ds[0]['text'][:1000])
print("...")

In [ ]:
# Step 8: Save Dataset Splits Locally
data_dir = os.path.join(PROJECT_ROOT, "data")
save_splits(train_ds, test_ds, output_dir=data_dir)